In [1]:
!pip install captum --quiet

zsh:1: command not found: pip


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from transformers import AutoModel, AutoTokenizer
from captum.attr import IntegratedGradients, LayerIntegratedGradients

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


In [4]:
import os

class CodeReviewModel(nn.Module):
    def __init__(self, model_name, num_labels=2, dropout=0.3):
        super().__init__()
        self.encoder    = AutoModel.from_pretrained(model_name)
        hidden_size     = self.encoder.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(hidden_size, 256),
            nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, num_labels)
        )

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return self.classifier(out.last_hidden_state[:, 0, :])


tokenizer = AutoTokenizer.from_pretrained('tokenizer/')
model     = CodeReviewModel('microsoft/codebert-base').to(DEVICE)

checkpoint_candidates = [
    '/Users/jin/Desktop/projects/Intelligent-Code-Review-Assistant-Using-Deep-Learning/code-review-ai/.gitignore/checkpoints/best_model.pt',
    '/Users/jin/Desktop/projects/Intelligent-Code-Review-Assistant-Using-Deep-Learning/code-review-ai/checkpoints/best_model.pt',
    '/Users/jin/Desktop/projects/Intelligent-Code-Review-Assistant-Using-Deep-Learning/code-review-ai/notebooks/checkpoints/best_model.pt',
]
checkpoint_path = next((path for path in checkpoint_candidates if os.path.exists(path)), None)

if checkpoint_path is None:
    raise FileNotFoundError(
        'Could not find best_model.pt. Tried: ' + ', '.join(checkpoint_candidates)
    )

model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
model.eval()
print(f"✅ Model loaded from {checkpoint_path}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 42485.31it/s]



✅ Model loaded from /Users/jin/Desktop/projects/Intelligent-Code-Review-Assistant-Using-Deep-Learning/code-review-ai/.gitignore/checkpoints/best_model.pt


In [7]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from transformers import AutoModel, AutoTokenizer


class CodeDefectDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=512):
        self.data       = dataframe.reset_index(drop=True)
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        code  = str(self.data.loc[idx, 'clean_code'])
        label = int(self.data.loc[idx, 'target'])
        enc   = self.tokenizer(code, max_length=self.max_length,
                               padding='max_length', truncation=True,
                               return_tensors='pt')
        return {
            'input_ids'      : enc['input_ids'].squeeze(0),
            'attention_mask' : enc['attention_mask'].squeeze(0),
            'label'          : torch.tensor(label, dtype=torch.long)
        }


class CodeReviewModel(nn.Module):
    def __init__(self, model_name, num_labels=2, dropout=0.3):
        super().__init__()
        self.encoder    = AutoModel.from_pretrained(model_name)
        hidden_size     = self.encoder.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(hidden_size, 256),
            nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, num_labels)
        )

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return self.classifier(out.last_hidden_state[:, 0, :])


tokenizer = AutoTokenizer.from_pretrained('tokenizer/')
model     = CodeReviewModel('microsoft/codebert-base').to(DEVICE)

checkpoint_candidates = [
    '/Users/jin/Desktop/projects/Intelligent-Code-Review-Assistant-Using-Deep-Learning/code-review-ai/.gitignore/checkpoints/best_model.pt',
    '/Users/jin/Desktop/projects/Intelligent-Code-Review-Assistant-Using-Deep-Learning/code-review-ai/checkpoints/best_model.pt',
    '/Users/jin/Desktop/projects/Intelligent-Code-Review-Assistant-Using-Deep-Learning/code-review-ai/notebooks/checkpoints/best_model.pt',
]
checkpoint_path = next((path for path in checkpoint_candidates if os.path.exists(path)), None)

if checkpoint_path is None:
    raise FileNotFoundError(
        'Could not find best_model.pt. Tried: ' + ', '.join(checkpoint_candidates)
    )

model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
model.eval()
print(f"✅ Model loaded from {checkpoint_path}")


def predict(code: str, threshold: float = 0.5):
    """Return prediction and confidence for a code snippet."""
    enc = tokenizer(
        code,
        max_length=512,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    input_ids      = enc['input_ids'].to(DEVICE)
    attention_mask = enc['attention_mask'].to(DEVICE)

    model.eval()
    with torch.no_grad():
        logits = model(input_ids, attention_mask)
        probs  = F.softmax(logits, dim=1).squeeze(0)

    clean_prob = probs[0].item()
    defect_prob = probs[1].item()
    predicted_label = 'DEFECTIVE 🔴' if defect_prob >= threshold else 'CLEAN ✅'
    predicted_prob = defect_prob if defect_prob >= threshold else clean_prob

    return {
        'prediction': predicted_label,
        'defect_prob': defect_prob,
        'clean_prob': clean_prob,
        'confidence': predicted_prob,
        'threshold': threshold,
    }


# Test on a known defective snippet (buffer overflow risk)
test_code = """
void copy_data(char *src) {
    char buf[10];
    strcpy(buf, src);  /* no bounds check — classic buffer overflow */
    printf("%s\\n", buf);
}
"""
result = predict(test_code)
print("=== Prediction ===")
for k, v in result.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10107.61it/s]



✅ Model loaded from /Users/jin/Desktop/projects/Intelligent-Code-Review-Assistant-Using-Deep-Learning/code-review-ai/.gitignore/checkpoints/best_model.pt
=== Prediction ===
  prediction: CLEAN ✅
  defect_prob: 0.4267
  clean_prob: 0.5733
  confidence: 0.5733
  threshold: 0.5000
=== Prediction ===
  prediction: CLEAN ✅
  defect_prob: 0.4267
  clean_prob: 0.5733
  confidence: 0.5733
  threshold: 0.5000


In [10]:
def get_token_attributions(code: str, target_class: int = 1):
    """
    Use Integrated Gradients on the embedding layer to score how much each token contributes to the prediction.
    Returns: (tokens list, attribution scores list)
    """
    enc = tokenizer(
        code,
        max_length=512,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    input_ids      = enc['input_ids'].to(DEVICE)
    attention_mask = enc['attention_mask'].to(DEVICE)

    def forward_from_embeddings(embeddings):
        expanded_mask = attention_mask
        if expanded_mask.size(0) == 1 and embeddings.size(0) != 1:
            expanded_mask = expanded_mask.expand(embeddings.size(0), -1)
        out = model.encoder(inputs_embeds=embeddings, attention_mask=expanded_mask)
        logits = model.classifier(out.last_hidden_state[:, 0, :])
        return F.softmax(logits, dim=1)[:, target_class]

    embeddings = model.encoder.embeddings.word_embeddings(input_ids)
    baseline   = torch.zeros_like(embeddings)

    ig = IntegratedGradients(forward_from_embeddings)
    attributions, _ = ig.attribute(
        embeddings,
        baselines=baseline,
        n_steps=25,
        return_convergence_delta=True,
        internal_batch_size=1,
    )

    attr_scores = attributions.sum(dim=-1).squeeze(0).cpu().detach().numpy()

    real_len   = attention_mask.sum().item()
    token_ids  = input_ids[0, :real_len].cpu().numpy()
    tokens     = tokenizer.convert_ids_to_tokens(token_ids)
    scores     = attr_scores[:real_len]

    return tokens, scores


tokens, scores = get_token_attributions(test_code, target_class=1)
print("Top-10 most attributed tokens:")
top_idx = np.argsort(np.abs(scores))[-10:][::-1]
for i in top_idx:
    print(f"  '{tokens[i]}'  score={scores[i]:.4f}")

Top-10 most attributed tokens:
  '</s>'  score=-0.0901
  '}'  score=-0.0229
  'Ċ'  score=-0.0092
  '['  score=0.0070
  '('  score=0.0068
  '('  score=0.0063
  'Ċ'  score=0.0051
  ');'  score=0.0047
  '",'  score=-0.0039
  'Ċ'  score=0.0037
